# Multi-Platform Product Comparison with Scavio API

Compare a product across Amazon, Walmart, and Google to find the best deal. Uses the Scavio search API and LangChain to pull live pricing, ratings, and availability from multiple platforms in one query.

**What you will learn:**
- Search products across Amazon, Walmart, and Google simultaneously
- Pull detailed product data from each platform
- Build a side-by-side comparison table
- Identify the best deal across platforms

**Prerequisites:**
- Free Scavio API key (250 credits/month): https://dashboard.scavio.dev
- OpenAI API key

**Tools used:** ScavioAmazonSearch, ScavioAmazonProduct, ScavioWalmartSearch, ScavioWalmartProduct, ScavioSearch

In [1]:
# pip install langchain langchain-openai langchain-scavio python-dotenv

In [2]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_scavio import (
    ScavioAmazonSearch,
    ScavioAmazonProduct,
    ScavioWalmartSearch,
    ScavioWalmartProduct,
    ScavioSearch,
)

load_dotenv(override=True)

True

In [3]:
SYSTEM_PROMPT = """You are ProductComparator, a multi-platform price comparison agent.

Workflow:
1. Take the user's product name.
2. Call ScavioAmazonSearch to find the product on Amazon.
3. Call ScavioAmazonProduct on the best match to get full Amazon details.
4. Call ScavioWalmartSearch to find the same product on Walmart.
5. Call ScavioWalmartProduct on the best match to get full Walmart details.
6. Call ScavioSearch for "<product name> best price" to see Google
   Shopping results and additional retailers.
7. Compile a comparison report:

   ## Product Comparison: <product name>

   ### Side-by-Side
   | Platform | Price | Rating | Reviews | Availability |
   |----------|-------|--------|---------|-------------|
   | Amazon   | $X.XX | X.X    | X,XXX   | <status>    |
   | Walmart  | $X.XX | X.X    | X,XXX   | <status>    |

   ### Price Difference
   - Savings: $X.XX by buying on <platform>
   - Percentage difference: X%

   ### Platform Pros/Cons
   - Amazon: <delivery speed, Prime benefits, return policy>
   - Walmart: <pickup options, rollback pricing, delivery>

   ### Verdict
   Buy on <platform> because <reason>.

Rules:
- Never invent prices, ratings, or availability. Only use tool output.
- Call only ONE tool per step.
- If a product is not found on one platform, say so.
- Keep the final report under 300 words.
"""

In [4]:
def build_agent():
    model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
    tools = [
        ScavioAmazonSearch(max_results=3),
        ScavioAmazonProduct(),
        ScavioWalmartSearch(max_results=3),
        ScavioWalmartProduct(),
        ScavioSearch(max_results=3),
    ]
    return create_agent(model, tools=tools, system_prompt=SYSTEM_PROMPT)

In [5]:
agent = build_agent()
result = agent.invoke({
    "messages": [{"role": "user", "content": "Sony WH-1000XM5 headphones"}]
})
print(result["messages"][-1].content)

## Product Comparison: Sony WH-1000XM5 headphones

### Side-by-Side
| Platform | Price | Rating | Reviews | Availability |
|----------|-------|--------|---------|--------------|
| Amazon   | $248  | 4.2    | 19,527  | In Stock     |
| Walmart  | $259  | N/A    | 0       | In Stock     |

### Price Difference
- Savings: $11 by buying on Amazon
- Percentage difference: ~4.2%

### Platform Pros/Cons
- Amazon: Offers Prime delivery with free shipping for Prime members, 30-day free return policy, and a large number of reviews for confidence in purchase.
- Walmart: Offers shipping and pickup options, but no rating or reviews available for this product. Slightly higher price than Amazon.

### Verdict
Buy on Amazon because it offers a lower price, Prime delivery benefits, a large number of customer reviews, and a flexible return policy.


## Next Steps

- Compare any product across Amazon and Walmart
- Track price differences over time to find seasonal deals
- Build a shopping assistant that always finds the cheapest platform
- Add more platforms as Scavio expands its API coverage

**Credits used:** ~5-8 per run (searches + product lookups on both platforms + Google)